In [1]:
%pip install ollama


[notice] A new release of pip is available: 25.1.1 -> 25.2
[notice] To update, run: pip install --upgrade pip
Note: you may need to restart the kernel to use updated packages.


In [ ]:
import os
import re
import csv
import time
import logging
from typing import List, Dict, Any
from concurrent.futures import ThreadPoolExecutor, as_completed

import pandas as pd
import ollama  # pip install ollama

# =========================
# Configuration
# =========================
model_name = "gemma3:12b"
input_file = "/data/gregIB/issuebench/2_final_dataset/combined_prompts_issues_with_topics.csv"

safe_model_name = re.sub(r'[:/\\]', '_', model_name)
output_file = f"//data/gregIB/issuebench/3_experiments/2_inference/completions/020925{safe_model_name}_completions.csv"

# Processing parameters
TEST_SUBSET = 20         # e.g., 200 to test first 200 rows; None = all
MAX_WORKERS = 2           # Tip: 1–3 often best for a single GPU model; keep if measured good
RETRY_ATTEMPTS = 1
RETRY_BACKOFF_SECS = 2
FLUSH_EVERY = 100          # how many completed rows before appending to disk

# ======= Requested generation settings =======
MAX_TOKENS = 1064          # <-- per your request
TEMPERATURE = 1.0          # <-- per your request
TOP_P = 0.9
NUM_BATCH = 256            # <-- per your request
KEEP_ALIVE = "1h"
CACHE_PROMPT = True

# Internal constants
ROW_ID_COL = "__row_id__"

logging.basicConfig(level=logging.INFO, format="%(asctime)s | %(levelname)s | %(message)s")
log = logging.getLogger("ollama-run")
for noisy in ("httpx", "httpcore", "urllib3", "ollama"):
    logging.getLogger(noisy).setLevel(logging.ERROR)

# this is surely slowing this down
def load_completed_row_ids(path: str) -> set:
    if not os.path.exists(path):
        return set()
    try:
        done = pd.read_csv(path, usecols=[ROW_ID_COL])
        return set(done[ROW_ID_COL].astype(int).tolist())
    except Exception:
        return set()

def append_rows(path: str, rows: List[Dict[str, Any]], columns: List[str]) -> None:
    new_file = not os.path.exists(path)
    with open(path, "a", newline="", encoding="utf-8") as f:
        writer = csv.writer(f)
        if new_file:
            writer.writerow(columns)
        for r in rows:
            writer.writerow([r.get(col, "") for col in columns])

def query_ollama_local(model: str, prompt: str) -> str:
    """Call Ollama locally using the ollama Python library."""
    try:
        resp = ollama.generate(
            model=model,
            prompt=prompt,
            stream=False,
            options={
                "temperature": TEMPERATURE,
                "top_p": TOP_P,
                "num_predict": MAX_TOKENS,   # requested cap
                "num_batch": NUM_BATCH,      # requested batch size
                "cache_prompt": CACHE_PROMPT,
            },
            keep_alive=KEEP_ALIVE,
        )
        return resp["response"].strip()
    except Exception as e:
        raise RuntimeError(f"Ollama local call failed: {e}")

def complete_with_retries(prompt: str) -> str:
    """Retry wrapper around query_ollama_local with simple backoff (no warnings, only final error)."""
    last_err = None
    for attempt in range(RETRY_ATTEMPTS + 1):
        try:
            return query_ollama_local(model_name, prompt)
        except Exception as e:
            last_err = e
            if attempt < RETRY_ATTEMPTS:
                time.sleep(RETRY_BACKOFF_SECS * (attempt + 1))
            else:
                raise last_err

def process_all_parallel(todo_idx: List[int], df: pd.DataFrame, out_cols: List[str]) -> None:
    """
    Fire all requests (bounded by MAX_WORKERS), update df in-memory,
    and append completed rows to disk in batches to avoid O(N) rewrites.
    """
    prompts = {i: str(df.at[i, "prompt_text"]) for i in todo_idx}
    total = len(todo_idx)
    completed = 0
    start = time.perf_counter()

    buffer_for_disk: List[Dict[str, Any]] = []

    log.info(
        f"Starting processing of {total} items with {MAX_WORKERS} workers | "
        f"MAX_TOKENS={MAX_TOKENS} | TEMPERATURE={TEMPERATURE} | NUM_BATCH={NUM_BATCH}"
    )

    with ThreadPoolExecutor(max_workers=MAX_WORKERS) as ex:
        future_to_idx = {ex.submit(complete_with_retries, prompt): i for i, prompt in prompts.items()}

        for fut in as_completed(future_to_idx):
            i = future_to_idx[fut]
            try:
                resp = fut.result()
            except Exception as e:
                resp = f"FutureError: {e}"
                log.error(f"Row {i}: {e}")

            # Update in-memory
            df.at[i, "response_text"] = resp
            df.at[i, "model"] = model_name

            # Stage a full row for append (so output file is a growing, valid CSV)
            row_dict = {col: df.at[i, col] if col in df.columns else "" for col in out_cols}
            buffer_for_disk.append(row_dict)

            completed += 1

            # Lightweight progress + ETA logging
            if completed % 5 == 0 or completed == total:
                elapsed = time.perf_counter() - start
                avg = elapsed / completed if completed else 0.0
                remaining = total - completed
                eta_min = (remaining * avg) / 60 if avg > 0 else 0.0
                pct = 100 * completed / total if total else 100.0
                log.info(f"Progress: {completed}/{total} ({pct:.1f}%) | Avg: {avg:.1f}s | ETA: {eta_min:.1f}min")

            # Append to disk in batches (no full-DataFrame rewrites)
            if len(buffer_for_disk) >= FLUSH_EVERY or completed == total:
                append_rows(output_file, buffer_for_disk, out_cols)
                buffer_for_disk.clear()

def main():
    t0 = time.perf_counter()
    df = pd.read_csv(input_file)

    # Ensure required columns
    if "response_text" not in df.columns:
        df["response_text"] = ""
    if "model" not in df.columns:
        df["model"] = ""

    # Stable row identifier for resumability
    df[ROW_ID_COL] = df.index.astype(int)

    # Output columns: helper id first for easy resume
    out_cols = [ROW_ID_COL] + [c for c in df.columns if c != ROW_ID_COL]

    # Resume support: skip rows already written
    already_done = load_completed_row_ids(output_file)

    # Todos from input that aren't already emitted
    mask_todo = df["response_text"].isna() | (df["response_text"].astype(str).str.strip() == "")
    todo_idx = [i for i in df.index[mask_todo].tolist() if i not in already_done]

    if TEST_SUBSET is not None:
        todo_idx = todo_idx[:TEST_SUBSET]

    total_rows = len(df)
    total_todos = len(todo_idx)

    log.info(f"Rows total = {total_rows} | to-complete (after resume check) = {total_todos} | model = {model_name}")
    log.info(f"Output file (append-only): {output_file}")

    if not todo_idx:
        log.info("Nothing to do. Exiting.")
        return

    process_all_parallel(todo_idx, df, out_cols)

    elapsed = time.perf_counter() - t0
    rate = total_todos / elapsed if elapsed > 0 else 0.0
    log.info(f"Done. Appended to: {output_file} | processed {total_todos} rows in {elapsed:.1f}s | Avg rate: {rate:.2f} req/s")

if __name__ == "__main__":
    main()


2025-09-03 10:05:11,561 | INFO | Rows total = 62178 | to-complete (after resume check) = 20 | model = gemma3:12b
2025-09-03 10:05:11,562 | INFO | Output file (append-only): //data/gregIB/issuebench/3_experiments/2_inference/completions/020925gemma3_12b_completions.csv
2025-09-03 10:05:11,563 | INFO | Starting processing of 20 items with 2 workers | MAX_TOKENS=1064 | TEMPERATURE=1.0 | NUM_BATCH=256
/tmp/ipykernel_8671/805094245.py:133: FutureWarning: Setting an item of incompatible dtype is deprecated and will raise an error in a future version of pandas. Value 'Okay, here are a few options for an introduction to a paper on cultural mythologies, ranging in tone and approach. I've included explanations after each option to help you choose the best fit for your paper's specific focus and intended audience.  **Please read the notes at the very end - they're crucial for adapting this to *your* paper.**

**Option 1: Broad & Engaging (Good for a general audience, sets the stage)**

> From the

KeyboardInterrupt: 

In [6]:
import os
import re
import csv
import time
import logging
from typing import List, Dict, Any
from concurrent.futures import ThreadPoolExecutor, as_completed

import pandas as pd
import ollama

# =========================
# Configuration
# =========================
model_name = "gemma3:12b"
input_file = "/data/gregIB/issuebench/2_final_dataset/combined_prompts_issues_with_topics.csv"
safe_model_name = re.sub(r'[:/\\]', '_', model_name)
output_file = f"/data/gregIB/issuebench/3_experiments/2_inference/completions/020925_{safe_model_name}_completions.csv"

# Processing parameters
TEST_SUBSET = 100         # None = all
MAX_WORKERS = 12            # Critical change: Reduced for GPU efficiency
RETRY_ATTEMPTS = 1
RETRY_BACKOFF_SECS = 2
FLUSH_EVERY = 100

# Generation settings
MAX_TOKENS = 1064
TEMPERATURE = 1.0
TOP_P = 0.9
NUM_BATCH = 256
KEEP_ALIVE = "1h"
CACHE_PROMPT = True

# Logging setup
logging.basicConfig(level=logging.INFO, format="%(asctime)s | %(levelname)s | %(message)s")
log = logging.getLogger("ollama-run")
for noisy in ("httpx", "httpcore", "urllib3", "ollama"):
    logging.getLogger(noisy).setLevel(logging.ERROR)

def append_rows(path: str, rows: List[Dict[str, Any]], columns: List[str]) -> None:
    new_file = not os.path.exists(path)
    with open(path, "a", newline="", encoding="utf-8") as f:
        writer = csv.writer(f)
        if new_file:
            writer.writerow(columns)
        for r in rows:
            writer.writerow([r.get(col, "") for col in columns])

def query_ollama_local(prompt: str) -> str:
    """Call Ollama with error handling"""
    try:
        resp = ollama.generate(
            model=model_name,
            prompt=prompt,
            stream=False,
            options={
                "temperature": TEMPERATURE,
                "top_p": TOP_P,
                "num_predict": MAX_TOKENS,
                "num_batch": NUM_BATCH,
                "cache_prompt": CACHE_PROMPT,
            },
            keep_alive=KEEP_ALIVE,
        )
        return resp["response"].strip()
    except Exception as e:
        raise RuntimeError(f"Ollama error: {e}")

def process_all_parallel(todo_idx: List[int], df: pd.DataFrame, out_cols: List[str]) -> None:
    buffer_for_disk = []
    start = time.perf_counter()
    total = len(todo_idx)
    
    with ThreadPoolExecutor(max_workers=MAX_WORKERS) as ex:
        future_map = {ex.submit(query_ollama_local, df.at[i, "prompt_text"]): i for i in todo_idx}
        
        for future in as_completed(future_map):
            i = future_map[future]
            try:
                response = future.result()
            except Exception as e:
                response = f"Error: {str(e)}"
                log.error(f"Failed row {i}: {e}")

            df.at[i, "response_text"] = response
            df.at[i, "model"] = model_name
            
            # Buffer for efficient I/O
            buffer_for_disk.append({col: df.at[i, col] for col in out_cols})
            
            # Progress tracking
            completed = len(buffer_for_disk)
            if completed % 5 == 0 or completed == total:
                elapsed = time.perf_counter() - start
                log.info(f"Processed {completed}/{total} ({completed/total:.1%}) | Avg: {elapsed/completed:.1f}s/req")

            # Flush buffer periodically
            if len(buffer_for_disk) >= FLUSH_EVERY or completed == total:
                append_rows(output_file, buffer_for_disk, out_cols)
                buffer_for_disk.clear()

def main():
    df = pd.read_csv(input_file)
    
    # Initialize columns if missing
    for col in ["response_text", "model"]:
        if col not in df.columns:
            df[col] = ""
    
    # Identify work items
    mask_todo = df["response_text"].isna() | (df["response_text"].astype(str).str.strip() == "")
    todo_idx = df.index[mask_todo].tolist()
    
    if TEST_SUBSET:
        todo_idx = todo_idx[:TEST_SUBSET]
    
    if not todo_idx:
        log.info("No rows need processing")
        return

    out_cols = [c for c in df.columns]
    process_all_parallel(todo_idx, df, out_cols)

if __name__ == "__main__":
    main()

/tmp/ipykernel_8671/1263744372.py:86: FutureWarning: Setting an item of incompatible dtype is deprecated and will raise an error in a future version of pandas. Value 'Okay, here's a beginning to a post on cultural mythologies, aiming for an engaging and accessible tone. I've included a few different options for length and style, so you can choose the one that best fits your overall post goals.  I'll also include some notes afterward about potential directions for the rest of the post.

**Option 1: Short & Hook-Focused (Good for Social Media/Introductory Blog)**

**(Image: A striking visual - perhaps a composite of different mythological figures, or a beautiful landscape associated with mythology.)**

**Beyond Fairy Tales: Why We Still Need Mythologies**

We all know the stories, right? Cinderella, Hercules, King Arthur… These tales feel like childhood memories, charming but distant. But what if I told you these stories – and countless others from cultures around the world – are so much

KeyboardInterrupt: 

In [ ]:
import pandas as pd
import ollama
import time
import re
from datetime import datetime, timedelta
from pathlib import Path
from typing import Optional
from tqdm.notebook import tqdm
import logging
from IPython.display import display, clear_output
import warnings
warnings.filterwarnings('ignore')

# config
model_name = "deepseek-r1:14b"
input_file = "/data/gregIB/issuebench/2_final_dataset/combined_prompts_issues_with_topics.csv"
safe_model_name = re.sub(r'[:/\\]', '*', model_name)
output_file = f"/data/gregIB/issuebench/3_experiments/2_inference/completions/020925*{safe_model_name}_completions.csv"

CHUNK_SIZE = 500  # Save progress every N rows
REQUEST_TIMEOUT = 120  # Timeout for individual requests (seconds)

def setup_logging():
    """Setup logging for notebook environment"""
    logging.basicConfig(
        level=logging.INFO,
        format='%(asctime)s - %(levelname)s - %(message)s',
        handlers=[
            logging.FileHandler(f'ollama_processing_{safe_model_name}_{datetime.now().strftime("%Y%m%d_%H%M%S")}.log'),
        ]
    )
    return logging.getLogger(__name__)

def call_ollama_model(prompt: str, model: str) -> Optional[str]:
    """
    Call Ollama model directly with error handling
    """
    try:
        response = ollama.generate(
            model=model,
            prompt=prompt,
            options={
                'temperature': 1,
                'top_p': 0.9,
                'num_predict': 512,  # Max tokens to generate
            }
        )
        return response['response'].strip()
        
    except Exception as e:
        print(f"Error generating response: {str(e)}")
        return None

def format_time(seconds):
    """Format seconds into readable time string"""
    if seconds < 60:
        return f"{seconds:.1f}s"
    elif seconds < 3600:
        return f"{seconds/60:.1f}m"
    else:
        return f"{seconds/3600:.1f}h"

def estimate_completion_time(avg_time_per_request: float, remaining_rows: int) -> str:
    """Calculate and format ETA"""
    if avg_time_per_request <= 0 or remaining_rows <= 0:
        return "Calculating..."
    
    total_seconds = avg_time_per_request * remaining_rows
    eta = datetime.now() + timedelta(seconds=total_seconds)
    
    return f"{format_time(total_seconds)} (ETA: {eta.strftime('%H:%M:%S')})"


class ProgressTracker:
    """for tracking and display"""
    
    def __init__(self, total_rows, start_idx=0):
        self.total_rows = total_rows
        self.processed_count = start_idx
        self.failed_count = 0
        self.times = []
        self.start_time = time.time()
        self.last_update = time.time()
        
    def update(self, row_time, failed=False):
        self.processed_count += 1
        self.times.append(row_time)
        
        if failed:
            self.failed_count += 1
        
        # Keep only last 50 times for rolling average
        if len(self.times) > 50:
            self.times.pop(0)
        
        # update every 5 rows or every 10 seconds
        if (self.processed_count % 5 == 0) or (time.time() - self.last_update) > 10:
            self.display_progress()
            self.last_update = time.time()
    
    def display_progress(self):
        """Dcurrent progress"""
        if not self.times:
            return
            
        avg_time = sum(self.times) / len(self.times)
        remaining = self.total_rows - self.processed_count
        eta_str = estimate_completion_time(avg_time, remaining)
        progress_pct = (self.processed_count / self.total_rows) * 100
        elapsed = time.time() - self.start_time
        
        clear_output(wait=True)
        print(f"🚀 Processing Progress")
        print(f"─" * 50)
        print(f"Completed: {self.processed_count:,}/{self.total_rows:,} ({progress_pct:.1f}%)")
        print(f"Failed: {self.failed_count:,}")
        print(f"Avg time/request: {avg_time:.2f}s")
        print(f"Elapsed time: {format_time(elapsed)}")
        print(f"Remaining time: {eta_str}")
        print(f"─" * 50)

def process_csv_with_ollama(input_path: str, output_path: str, model: str, 
                           test_rows: Optional[int] = None, resume: bool = True):
    """
    Main processing function for Jupyter notebook
    """
    logger = setup_logging()
    
    print(f"Configuration")
    print(f"Model: {model}")
    print(f"Input: {input_path}")
    print(f"Output: {output_path}")
    print(f"Test rows: {test_rows or 'All rows'}")
    print(f"Resume: {'Yes' if resume else 'No'}")
    print("="*60)
    
    
    print("="*60)
    
    # Load input data
    try:
        print("Loading input CSV...")
        df = pd.read_csv(input_path)
        print(f"✅ Loaded {len(df):,} rows")
        
            
    except Exception as e:
        print(f"❌ Error loading input file: {str(e)}")
        return False
    
    # Handle test subset
    if test_rows:
        df = df.head(test_rows)
        print(f"processing test subset of {len(df):,} rows")
    
    # Add model column
    df['model'] = ''
    
    # Check for existing output file and resume option
    start_idx = 0
    if resume and Path(output_path).exists():
        try:
            existing_df = pd.read_csv(output_path)
            completed_mask = existing_df['response_text'].notna() & (existing_df['response_text'] != '')
            completed_rows = completed_mask.sum()
            start_idx = completed_rows
            print(f"found existing output file")
            print(f"resuming from row {start_idx:,} ({completed_rows:,} completed rows)")
            
            # Update df with existing data
            min_rows = min(len(existing_df), len(df))
            for idx in range(min_rows):
                if completed_mask.iloc[idx]:
                    df.iloc[idx] = existing_df.iloc[idx]
                    
        except Exception as e:
            print(f"⚠️ Could not resume from existing file: {str(e)}. Starting from beginning...")
    
    print("="*60)
    
    # Initialize progress tracker
    total_rows = len(df)
    tracker = ProgressTracker(total_rows, start_idx)
    
    print(f"starting processing from row {start_idx:,}/{total_rows:,}")
    print("="*60)
    
    # Process rows
    last_save_time = time.time()
    
    try:
        for idx in range(start_idx, total_rows):
            row_start_time = time.time()
            
            # Skip if already processed
            if pd.notna(df.iloc[idx]['response_text']) and str(df.iloc[idx]['response_text']).strip() != '':
                tracker.update(0.01, failed=False)  # Minimal time for skipped rows
                continue
            
            prompt = str(df.iloc[idx]['prompt_text'])
            
            # Skip empty prompts
            if not prompt or prompt.strip() == '' or prompt.lower() == 'nan':
                df.iloc[idx, df.columns.get_loc('response_text')] = '[EMPTY_PROMPT]'
                df.iloc[idx, df.columns.get_loc('model')] = model
                tracker.update(0.01, failed=True)
                continue
            
            # Call Ollama model
            response = call_ollama_model(prompt, model)
            
            if response is not None:
                df.iloc[idx, df.columns.get_loc('response_text')] = response
                df.iloc[idx, df.columns.get_loc('model')] = model
                failed = False
            else:
                df.iloc[idx, df.columns.get_loc('response_text')] = '[API_ERROR]'
                df.iloc[idx, df.columns.get_loc('model')] = model
                failed = True
                # Only print errors occasionally to avoid spam
                if tracker.failed_count % 10 == 0:
                    print(f"⚠️ API errors: {tracker.failed_count}")
            
            row_time = time.time() - row_start_time
            tracker.update(row_time, failed)
            
            # Save checkpoint
            if (tracker.processed_count % CHUNK_SIZE == 0) or (time.time() - last_save_time) > 300:
                try:
                    df.to_csv(output_path, index=False)
                    last_save_time = time.time()
                    print(f"checkpoint saved at row {tracker.processed_count:,}")
                except Exception as e:
                    print(f"failed to save checkpoint: {str(e)}")
    
    except KeyboardInterrupt:
        print("\n⏹️ Processing interrupted by user")
        print("saving current progress...")
        df.to_csv(output_path, index=False)
        return False
    
    # Final save and summary
    try:
        df.to_csv(output_path, index=False)
        clear_output(wait=True)
        
        print("processing Complete!")
        print("="*60)
        print(f"output saved to: {output_path}")
        print(f"total processed: {tracker.processed_count:,}/{total_rows:,}")
        print(f"❌ failed requests: {tracker.failed_count:,}")
        
        if tracker.times:
            avg_time = sum(tracker.times) / len(tracker.times)
            total_time = time.time() - tracker.start_time
            print(f"avg time/request: {avg_time:.2f}s")
            print(f"total processing time: {format_time(total_time)}")
            print(f"{len(tracker.times)/total_time*3600:.0f} requests/hour")
        
        return True
        
    except Exception as e:
        print(f"❌ Failed to save final output: {str(e)}")
        return False

# Convenience functions for notebook usage
def run_test(rows=10):
    """Quick test run with specified number of rows"""
    return process_csv_with_ollama(
        input_path=input_file,
        output_path=output_file.replace('.csv', '_TEST.csv'),
        model=model_name,
        test_rows=rows,
        resume=False
    )

def run_full():
    """Run full processing"""
    return process_csv_with_ollama(
        input_path=input_file,
        output_path=output_file,
        model=model_name,
        test_rows=None,
        resume=False
    )

run_full()

🚀 Processing Progress
──────────────────────────────────────────────────
Completed: 2/62,178 (0.0%)
Failed: 0
Avg time/request: 18.31s
Elapsed time: 36.6s
Remaining time: 316.3h (ETA: 15:43:30)
──────────────────────────────────────────────────

⏹️ Processing interrupted by user
saving current progress...


False

In [ ]:
# CSV Ollama Batch Processor for Jupyter Notebook
# Optimized for high-throughput processing with parallel requests

import pandas as pd
import ollama
import time
import re
from datetime import datetime, timedelta
from pathlib import Path
from typing import Optional, List
from tqdm.notebook import tqdm
import logging
from IPython.display import display, clear_output
import warnings
import concurrent.futures
import threading
from threading import Lock
import queue

warnings.filterwarnings('ignore')

# Configuration
model_name = "deepseek-r1:14b"
input_file = "/data/gregIB/issuebench/2_final_dataset/combined_prompts_issues_with_topics.csv"
safe_model_name = re.sub(r'[:/\\]', '*', model_name)
output_file = f"/data/gregIB/issuebench/3_experiments/2_inference/completions/020925*{safe_model_name}_completions.csv"

# Performance tuning parameters
CHUNK_SIZE = 1000  # Increased chunk size for fewer I/O operations
REQUEST_TIMEOUT = 120  # Timeout for individual requests (seconds)
MAX_WORKERS = 4  # Reduced number of workers to avoid overloading
RETRY_ATTEMPTS = 3  # Number of retry attempts for failed requests

def setup_logging():
    """Setup logging for notebook environment"""
    logging.basicConfig(
        level=logging.INFO,
        format='%(asctime)s - %(levelname)s - %(message)s',
        handlers=[
            logging.FileHandler(f'ollama_processing_{safe_model_name}_{datetime.now().strftime("%Y%m%d_%H%M%S")}.log'),
        ]
    )
    return logging.getLogger(__name__)

def call_ollama_model(prompt: str, model: str) -> Optional[str]:
    """
    Call Ollama model directly with error handling and retries
    """
    for attempt in range(RETRY_ATTEMPTS):
        try:
            response = ollama.generate(
                model=model,
                prompt=prompt,
                options={
                    'temperature': 1,
                    'top_p': 0.9,
                    'num_predict': 512,
                }
            )
            return response['response'].strip()
            
        except Exception as e:
            print(f"Error generating response (attempt {attempt+1}/{RETRY_ATTEMPTS}): {str(e)}")
            if attempt == RETRY_ATTEMPTS - 1:
                return f"[API_ERROR_FINAL]"
            time.sleep(2 ** attempt)  # Exponential backoff

def process_single_item(item):
    """Process a single prompt with its index"""
    idx, prompt, model = item
    return idx, call_ollama_model(prompt, model)

def format_time(seconds):
    """Format seconds into readable time string"""
    if seconds < 60:
        return f"{seconds:.1f}s"
    elif seconds < 3600:
        return f"{seconds/60:.1f}m"
    else:
        return f"{seconds/3600:.1f}h"

def estimate_completion_time(avg_time_per_request: float, remaining_rows: int) -> str:
    """Calculate and format ETA"""
    if avg_time_per_request <= 0 or remaining_rows <= 0:
        return "Calculating..."
    
    total_seconds = avg_time_per_request * remaining_rows
    eta = datetime.now() + timedelta(seconds=total_seconds)
    
    return f"{format_time(total_seconds)} (ETA: {eta.strftime('%H:%M:%S')})"

class ProgressTracker:
    """Class to handle progress tracking and display in Jupyter"""
    
    def __init__(self, total_rows, start_idx=0):
        self.total_rows = total_rows
        self.processed_count = start_idx
        self.failed_count = 0
        self.times = []
        self.start_time = time.time()
        self.last_update = time.time()
        self.lock = Lock()
        
    def update(self, processed_delta=1, failed_delta=0, processing_time=0):
        """Update progress with thread-safe locking"""
        with self.lock:
            self.processed_count += processed_delta
            self.failed_count += failed_delta
            
            if processing_time > 0:
                self.times.append(processing_time)
                # Keep only last 50 times for rolling average
                if len(self.times) > 50:
                    self.times.pop(0)
            
            # Update display every 5 rows or every 10 seconds
            current_time = time.time()
            if (self.processed_count % 5 == 0) or (current_time - self.last_update) > 10:
                self.display_progress()
                self.last_update = current_time
    
    def display_progress(self):
        """Display current progress"""
        if not self.times:
            return
            
        avg_time = sum(self.times) / len(self.times) if self.times else 0
        remaining = self.total_rows - self.processed_count
        eta_str = estimate_completion_time(avg_time, remaining)
        progress_pct = (self.processed_count / self.total_rows) * 100
        elapsed = time.time() - self.start_time
        
        clear_output(wait=True)
        print(f"🚀 Processing Progress (Parallel: {MAX_WORKERS} workers)")
        print(f"─" * 60)
        print(f"Completed: {self.processed_count:,}/{self.total_rows:,} ({progress_pct:.1f}%)")
        print(f"Failed: {self.failed_count:,}")
        print(f"Avg time/request: {avg_time:.2f}s")
        print(f"Current rate: {1/avg_time if avg_time > 0 else 0:.2f} req/s")
        print(f"Elapsed time: {format_time(elapsed)}")
        print(f"Remaining time: {eta_str}")
        print(f"─" * 60)

def process_csv_with_ollama_parallel(input_path: str, output_path: str, model: str, 
                                   test_rows: Optional[int] = None, resume: bool = True):
    """
    Main processing function for Jupyter notebook with parallel processing
    """
    logger = setup_logging()
    
    print(f"Configuration")
    print(f"Model: {model}")
    print(f"Input: {input_path}")
    print(f"Output: {output_path}")
    print(f"Workers: {MAX_WORKERS}")
    print(f"Test rows: {test_rows or 'All rows'}")
    print(f"Resume: {'Yes' if resume else 'No'}")
    print("="*60)
    
    # Load input data
    try:
        print("Loading input CSV...")
        df = pd.read_csv(input_path)
        print(f"✅ Loaded {len(df):,} rows")
    except Exception as e:
        print(f"❌ Error loading input file: {str(e)}")
        return False
    
    # Handle test subset
    if test_rows:
        df = df.head(test_rows)
        print(f"Processing test subset of {len(df):,} rows")
    
    # Add model column if it doesn't exist
    if 'model' not in df.columns:
        df['model'] = ''
    
    # Check for existing output file and resume option
    start_idx = 0
    if resume and Path(output_path).exists():
        try:
            existing_df = pd.read_csv(output_path)
            completed_mask = existing_df['response_text'].notna() & (existing_df['response_text'] != '')
            completed_rows = completed_mask.sum()
            start_idx = completed_rows
            print(f"Found existing output file")
            print(f"Resuming from row {start_idx:,} ({completed_rows:,} completed rows)")
            
            # Update df with existing data
            min_rows = min(len(existing_df), len(df))
            for idx in range(min_rows):
                if completed_mask.iloc[idx]:
                    df.iloc[idx] = existing_df.iloc[idx]
                    
        except Exception as e:
            print(f"⚠️ Could not resume from existing file: {str(e)}. Starting from beginning...")
    
    print("="*60)
    
    # Initialize progress tracker
    total_rows = len(df)
    tracker = ProgressTracker(total_rows, start_idx)
    
    print(f"Starting parallel processing from row {start_idx:,}/{total_rows:,}")
    print("="*60)
    
    # Prepare items for processing
    items_to_process = []
    for idx in range(start_idx, total_rows):
        # Skip if already processed
        if pd.notna(df.iloc[idx]['response_text']) and str(df.iloc[idx]['response_text']).strip() != '':
            continue
        
        prompt = str(df.iloc[idx]['prompt_text'])
        
        # Skip empty prompts
        if not prompt or prompt.strip() == '' or prompt.lower() == 'nan':
            df.iloc[idx, df.columns.get_loc('response_text')] = '[EMPTY_PROMPT]'
            df.iloc[idx, df.columns.get_loc('model')] = model
            tracker.update(processed_delta=1)
            continue
        
        items_to_process.append((idx, prompt, model))
    
    print(f"Prepared {len(items_to_process):,} items for processing")
    
    # Process items using ThreadPoolExecutor
    last_save_time = time.time()
    processed_count = 0
    
    try:
        with concurrent.futures.ThreadPoolExecutor(max_workers=MAX_WORKERS) as executor:
            # Submit all tasks
            future_to_item = {executor.submit(process_single_item, item): item for item in items_to_process}
            
            # Process results as they complete
            for future in concurrent.futures.as_completed(future_to_item):
                item = future_to_item[future]
                idx, prompt, model = item
                
                try:
                    result_idx, response = future.result()
                    df.iloc[result_idx, df.columns.get_loc('response_text')] = response
                    df.iloc[result_idx, df.columns.get_loc('model')] = model
                    
                    # Check if response indicates failure
                    failed = response.startswith('[API_ERROR')
                    tracker.update(processed_delta=1, failed_delta=1 if failed else 0)
                    processed_count += 1
                    
                except Exception as e:
                    print(f"Error processing item {idx}: {str(e)}")
                    df.iloc[idx, df.columns.get_loc('response_text')] = f'[PROCESSING_ERROR: {str(e)}]'
                    df.iloc[idx, df.columns.get_loc('model')] = model
                    tracker.update(processed_delta=1, failed_delta=1)
                    processed_count += 1
                
                # Save checkpoint periodically
                if (processed_count % CHUNK_SIZE == 0) or (time.time() - last_save_time) > 300:
                    try:
                        df.to_csv(output_path, index=False)
                        last_save_time = time.time()
                        print(f"Checkpoint saved at {processed_count:,} processed items")
                    except Exception as e:
                        print(f"Failed to save checkpoint: {str(e)}")
    
    except KeyboardInterrupt:
        print("\n⏹️ Processing interrupted by user")
        print("Saving current progress...")
        df.to_csv(output_path, index=False)
        return False
    
    # Final save and summary
    try:
        df.to_csv(output_path, index=False)
        clear_output(wait=True)
        
        print("✅ Processing Complete!")
        print("="*60)
        print(f"Output saved to: {output_path}")
        print(f"Total processed: {tracker.processed_count:,}/{total_rows:,}")
        print(f"❌ Failed requests: {tracker.failed_count:,}")
        
        if tracker.times:
            total_time = time.time() - tracker.start_time
            avg_time = total_time / tracker.processed_count if tracker.processed_count else 0
            print(f"Avg time/request: {avg_time:.2f}s")
            print(f"Total processing time: {format_time(total_time)}")
            print(f"Processing rate: {tracker.processed_count/total_time*3600:.0f} requests/hour")
        
        return True
        
    except Exception as e:
        print(f"❌ Failed to save final output: {str(e)}")
        return False

# Convenience functions for notebook usage
def run_test(rows=10):
    """Quick test run with specified number of rows"""
    return process_csv_with_ollama_parallel(
        input_path=input_file,
        output_path=output_file.replace('.csv', '_TEST.csv'),
        model=model_name,
        test_rows=rows,
        resume=False
    )

def run_full():
    """Run full processing"""
    return process_csv_with_ollama_parallel(
        input_path=input_file,
        output_path=output_file,
        model=model_name,
        test_rows=None,
        resume=False
    )

# Run the processing
if __name__ == "__main__":
    run_full()

Configuration
Model: deepseek-r1:14b
Input: /data/gregIB/issuebench/2_final_dataset/combined_prompts_issues_with_topics.csv
Output: /data/gregIB/issuebench/3_experiments/2_inference/completions/020925*deepseek-r1*14b_completions.csv
Workers: 4
Test rows: All rows
Resume: No
Loading input CSV...
✅ Loaded 62,178 rows
Starting parallel processing from row 0/62,178
Prepared 62,178 items for processing
Checkpoint saved at 60 processed items


In [2]:
%pip install tqdm


[notice] A new release of pip is available: 25.1.1 -> 25.2
[notice] To update, run: pip install --upgrade pip
Note: you may need to restart the kernel to use updated packages.


In [ ]:
import pandas as pd
import ollama
import time
import re
from datetime import datetime
from pathlib import Path
from typing import Optional
import warnings

warnings.filterwarnings('ignore')

# Configuration
model_name = "deepseek-r1:14b"
input_file = "/data/gregIB/issuebench/2_final_dataset/combined_prompts_issues_with_topics.csv"
safe_model_name = re.sub(r'[:/\\]', '*', model_name)
output_file = f"/data/gregIB/issuebench/3_experiments/2_inference/completions/020925*{safe_model_name}_completions.csv"

# Performance tuning parameters
CHUNK_SIZE = 5000  # Larger chunk size for fewer I/O operations
REQUEST_TIMEOUT = 120  # Timeout for individual requests (seconds)

def call_ollama_model(prompt: str, model: str) -> Optional[str]:
    """Call Ollama model without retry logic"""
    try:
        response = ollama.generate(
            model=model,
            prompt=prompt,
            options={
                'temperature': 1,
                'top_p': 0.9,
                'num_predict': 512,
            }
        )
        return response['response'].strip()
    except Exception as e:
        return f"[API_ERROR: {str(e)}]"

def process_chunk(df_chunk, model, start_idx):
    """Process a chunk of prompts"""
    results = []
    for idx, row in df_chunk.iterrows():
        prompt = str(row['prompt_text'])
        
        # Skip empty prompts
        if not prompt or prompt.strip() == '' or prompt.lower() == 'nan':
            results.append((start_idx + idx, '[EMPTY_PROMPT]'))
            continue
            
        response = call_ollama_model(prompt, model)
        results.append((start_idx + idx, response))
    
    return results

def process_csv_with_ollama_optimized(input_path: str, output_path: str, model: str, 
                                   test_rows: Optional[int] = None, resume: bool = True):
    """Optimized processing function without retry logic"""
    print(f"Starting processing with model: {model}")
    
    # Load input data
    df = pd.read_csv(input_path)
    if test_rows:
        df = df.head(test_rows)
    
    total_rows = len(df)
    print(f"Processing {total_rows:,} rows")
    
    # Initialize output columns if they don't exist
    if 'response_text' not in df.columns:
        df['response_text'] = None
    if 'model' not in df.columns:
        df['model'] = None
    
    # Determine start index for resuming
    start_idx = 0
    if resume and Path(output_path).exists():
        try:
            existing_df = pd.read_csv(output_path)
            completed_mask = existing_df['response_text'].notna() & (existing_df['response_text'] != '')
            start_idx = completed_mask.sum()
            print(f"Resuming from row {start_idx:,}")
        except Exception as e:
            print(f"Could not resume from existing file: {str(e)}")
    
    # Process data in chunks
    processed_count = start_idx
    start_time = time.time()
    
    for chunk_start in range(start_idx, total_rows, CHUNK_SIZE):
        chunk_end = min(chunk_start + CHUNK_SIZE, total_rows)
        chunk = df.iloc[chunk_start:chunk_end].copy()
        
        print(f"Processing rows {chunk_start:,} to {chunk_end-1:,}...")
        
        # Process the chunk
        chunk_results = process_chunk(chunk, model, chunk_start)
        
        # Update the DataFrame with results
        for idx, response in chunk_results:
            df.at[idx, 'response_text'] = response
            df.at[idx, 'model'] = model
        
        # Update progress
        processed_count += len(chunk)
        elapsed_time = time.time() - start_time
        rows_per_hour = processed_count / elapsed_time * 3600
        eta_hours = (total_rows - processed_count) / rows_per_hour if rows_per_hour > 0 else 0
        
        print(f"Processed: {processed_count:,}/{total_rows:,} "
              f"({processed_count/total_rows*100:.1f}%) "
              f"ETA: {eta_hours:.1f}h")
        
        # Save checkpoint
        df.to_csv(output_path, index=False)
        print(f"Checkpoint saved at row {chunk_end-1}")
    
    print("Processing complete!")
    return True

# Run the processing
if __name__ == "__main__":
    process_csv_with_ollama_optimized(
        input_path=input_file,
        output_path=output_file,
        model=model_name,
        test_rows=None,
        resume=True
    )

## BEST SO FAR

In [ ]:
# BEST SETUP SO FAR

import pandas as pd
import ollama
import time
import re
from datetime import datetime, timedelta
from pathlib import Path
from typing import Optional, List
from tqdm.auto import tqdm
import logging
from IPython.display import display, clear_output
import warnings
import concurrent.futures
import threading
from threading import Lock
import queue

warnings.filterwarnings('ignore')

# Configuration
model_name = "deepseek-r1:14b"
input_file = "/data/gregIB/issuebench/2_final_dataset/combined_prompts_issues_with_topics.csv"
safe_model_name = re.sub(r'[:/\\]', '*', model_name)
output_file = f"/data/gregIB/issuebench/3_experiments/2_inference/completions/020925*{safe_model_name}_completions.csv"

# Performance tuning parameters
CHUNK_SIZE = 1000  # Increased chunk size for fewer I/O operations
REQUEST_TIMEOUT = 120  # Timeout for individual requests (seconds)
MAX_WORKERS = 4  # Reduced number of workers to avoid overloading
RETRY_ATTEMPTS = 1  # Number of retry attempts for failed requests

def setup_logging():
    """Setup logging for notebook environment"""
    logging.basicConfig(
        level=logging.INFO,
        format='%(asctime)s - %(levelname)s - %(message)s',
        handlers=[
            logging.FileHandler(f'ollama_processing_{safe_model_name}_{datetime.now().strftime("%Y%m%d_%H%M%S")}.log'),
        ]
    )
    return logging.getLogger(__name__)

def call_ollama_model(prompt: str, model: str) -> Optional[str]:
    """
    Call Ollama model directly with error handling and retries
    """
    for attempt in range(RETRY_ATTEMPTS):
        try:
            response = ollama.generate(
                model=model,
                prompt=prompt,
                options={
                    'temperature': 1,
                    'top_p': 0.9,
                    'num_predict': 512,
                }
            )
            return response['response'].strip()
            
        except Exception as e:
            print(f"Error generating response (attempt {attempt+1}/{RETRY_ATTEMPTS}): {str(e)}")
            if attempt == RETRY_ATTEMPTS - 1:
                return f"[API_ERROR_FINAL]"
            time.sleep(2 ** attempt)  # Exponential backoff

def process_single_item(item):
    """Process a single prompt with its index"""
    idx, prompt, model = item
    start_time = time.time()
    result = call_ollama_model(prompt, model)
    processing_time = time.time() - start_time
    return idx, result, processing_time

def format_time(seconds):
    """Format seconds into readable time string"""
    if seconds < 60:
        return f"{seconds:.1f}s"
    elif seconds < 3600:
        return f"{seconds/60:.1f}m"
    else:
        return f"{seconds/3600:.1f}h"

class ProgressTracker:
    """Class to handle progress tracking and display in Jupyter"""
    
    def __init__(self, total_rows, start_idx=0):
        self.total_rows = total_rows
        self.processed_count = start_idx
        self.failed_count = 0
        self.times = []
        self.start_time = time.time()
        self.last_update = time.time()
        self.lock = Lock()
        self.progress_bar = tqdm(
            total=total_rows, 
            initial=start_idx,
            desc="Processing",
            unit="row",
            bar_format='{l_bar}{bar}| {n_fmt}/{total_fmt} [{elapsed}<{remaining}, {rate_fmt}]',
            leave=False  # Added parameter
        )
        
    def update(self, processed_delta=1, failed_delta=0, processing_time=0):
        """Update progress with thread-safe locking"""
        with self.lock:
            self.processed_count += processed_delta
            self.failed_count += failed_delta
            
            if processing_time > 0:
                self.times.append(processing_time)
                # Keep only last 50 times for rolling average
                if len(self.times) > 50:
                    self.times.pop(0)
            
            # Update progress bar
            self.progress_bar.update(processed_delta)
            
            # Update description with current stats
            if self.times:
                avg_time = sum(self.times) / len(self.times)
                remaining = self.total_rows - self.processed_count
                eta_seconds = avg_time * remaining
                
                # Format the description
                desc = f"Processed: {self.processed_count}/{self.total_rows} "
                desc += f"Failed: {self.failed_count} "
                desc += f"Avg: {avg_time:.2f}s "
                desc += f"ETA: {format_time(eta_seconds)}"
                
                self.progress_bar.set_description(desc)
            
            # Force display update every 10 rows
            if self.processed_count % 10 == 0:
                self.progress_bar.refresh()
    
    def close(self):
        """Close the progress bar safely"""
        try:
            self.progress_bar.close()
        except AttributeError:
            # Handle case where disp method might be missing
            pass
    
    def display_final_stats(self):
        """Display final statistics"""
        clear_output(wait=True)
        print("✅ Processing Complete!")
        print("="*60)
        print(f"Total processed: {self.processed_count:,}/{self.total_rows:,}")
        print(f"Failed requests: {self.failed_count:,}")
        
        if self.times:
            total_time = time.time() - self.start_time
            avg_time = total_time / self.processed_count if self.processed_count else 0
            print(f"Avg time/request: {avg_time:.2f}s")
            print(f"Total processing time: {format_time(total_time)}")
            print(f"Processing rate: {self.processed_count/total_time*3600:.0f} requests/hour")

def process_csv_with_ollama_parallel(input_path: str, output_path: str, model: str, 
                                   test_rows: Optional[int] = None, resume: bool = True):
    """
    Main processing function for Jupyter notebook with parallel processing
    """
    logger = setup_logging()
    
    print(f"Configuration")
    print(f"Model: {model}")
    print(f"Input: {input_path}")
    print(f"Output: {output_path}")
    print(f"Workers: {MAX_WORKERS}")
    print(f"Test rows: {test_rows or 'All rows'}")
    print(f"Resume: {'Yes' if resume else 'No'}")
    print("="*60)
    
    # Load input data
    try:
        print("Loading input CSV...")
        df = pd.read_csv(input_path)
        print(f"✅ Loaded {len(df):,} rows")
    except Exception as e:
        print(f"❌ Error loading input file: {str(e)}")
        return False
    
    # Handle test subset
    if test_rows:
        df = df.head(test_rows)
        print(f"Processing test subset of {len(df):,} rows")
    
    # Add model column if it doesn't exist
    if 'model' not in df.columns:
        df['model'] = ''
    
    # Check for existing output file and resume option
    start_idx = 0
    if resume and Path(output_path).exists():
        try:
            existing_df = pd.read_csv(output_path)
            completed_mask = existing_df['response_text'].notna() & (existing_df['response_text'] != '')
            completed_rows = completed_mask.sum()
            start_idx = completed_rows
            print(f"Found existing output file")
            print(f"Resuming from row {start_idx:,} ({completed_rows:,} completed rows)")
            
            # Update df with existing data
            min_rows = min(len(existing_df), len(df))
            for idx in range(min_rows):
                if completed_mask.iloc[idx]:
                    df.iloc[idx] = existing_df.iloc[idx]
                    
        except Exception as e:
            print(f"⚠️ Could not resume from existing file: {str(e)}. Starting from beginning...")
    
    print("="*60)
    
    # Initialize progress tracker
    total_rows = len(df)
    tracker = ProgressTracker(total_rows, start_idx)
    
    print(f"Starting parallel processing from row {start_idx:,}/{total_rows:,}")
    print("="*60)
    
    # Prepare items for processing
    items_to_process = []
    for idx in range(start_idx, total_rows):
        # Skip if already processed
        if pd.notna(df.iloc[idx]['response_text']) and str(df.iloc[idx]['response_text']).strip() != '':
            tracker.update(processed_delta=1)
            continue
        
        prompt = str(df.iloc[idx]['prompt_text'])
        
        # Skip empty prompts
        if not prompt or prompt.strip() == '' or prompt.lower() == 'nan':
            df.iloc[idx, df.columns.get_loc('response_text')] = '[EMPTY_PROMPT]'
            df.iloc[idx, df.columns.get_loc('model')] = model
            tracker.update(processed_delta=1)
            continue
        
        items_to_process.append((idx, prompt, model))
    
    print(f"Prepared {len(items_to_process):,} items for processing")
    
    # Process items using ThreadPoolExecutor
    last_save_time = time.time()
    
    try:
        with concurrent.futures.ThreadPoolExecutor(max_workers=MAX_WORKERS) as executor:
            # Submit all tasks
            future_to_item = {executor.submit(process_single_item, item): item for item in items_to_process}
            
            # Process results as they complete
            for future in concurrent.futures.as_completed(future_to_item):
                item = future_to_item[future]
                idx, prompt, model = item
                
                try:
                    result_idx, response, processing_time = future.result()
                    df.iloc[result_idx, df.columns.get_loc('response_text')] = response
                    df.iloc[result_idx, df.columns.get_loc('model')] = model
                    
                    # Check if response indicates failure
                    failed = response.startswith('[API_ERROR')
                    tracker.update(processed_delta=1, failed_delta=1 if failed else 0, processing_time=processing_time)
                    
                except Exception as e:
                    print(f"Error processing item {idx}: {str(e)}")
                    df.iloc[idx, df.columns.get_loc('response_text')] = f'[PROCESSING_ERROR: {str(e)}]'
                    df.iloc[idx, df.columns.get_loc('model')] = model
                    tracker.update(processed_delta=1, failed_delta=1)
                
                # Save checkpoint periodically
                if (tracker.processed_count % CHUNK_SIZE == 0) or (time.time() - last_save_time) > 300:
                    try:
                        df.to_csv(output_path, index=False)
                        last_save_time = time.time()
                        print(f"\nCheckpoint saved at {tracker.processed_count:,} processed items")
                    except Exception as e:
                        print(f"\nFailed to save checkpoint: {str(e)}")
    
    except KeyboardInterrupt:
        print("\n⏹️ Processing interrupted by user")
        tracker.close()
        print("Saving current progress...")
        df.to_csv(output_path, index=False)
        return False
    
    # Final save and summary
    try:
        tracker.close()
        df.to_csv(output_path, index=False)
        tracker.display_final_stats()
        print(f"Output saved to: {output_path}")
        return True
        
    except Exception as e:
        print(f"❌ Failed to save final output: {str(e)}")
        return False

# Convenience functions for notebook usage
def run_test(rows=10):
    """Quick test run with specified number of rows"""
    return process_csv_with_ollama_parallel(
        input_path=input_file,
        output_path=output_file.replace('.csv', '_TEST.csv'),
        model=model_name,
        test_rows=rows,
        resume=False
    )

def run_full():
    """Run full processing"""
    return process_csv_with_ollama_parallel(
        input_path=input_file,
        output_path=output_file,
        model=model_name,
        test_rows=None,
        resume=True
    )

# Run the processing
if __name__ == "__main__":
    run_full()

/data/gregIB/issuebench/.venv/lib/python3.12/site-packages/tqdm/auto.py:21: TqdmWarning: IProgress not found. Please update jupyter and ipywidgets. See https://ipywidgets.readthedocs.io/en/stable/user_install.html
  from .autonotebook import tqdm as notebook_tqdm


Configuration
Model: deepseek-r1:14b
Input: /data/gregIB/issuebench/2_final_dataset/combined_prompts_issues_with_topics.csv
Output: /data/gregIB/issuebench/3_experiments/2_inference/completions/020925*deepseek-r1*14b_completions.csv
Workers: 4
Test rows: All rows
Resume: Yes
Loading input CSV...
✅ Loaded 62,178 rows
Found existing output file
Resuming from row 60 (60 completed rows)


Processing:   0%|          | 60/62178 [00:00<?, ?row/s]

Starting parallel processing from row 60/62,178
Prepared 62,118 items for processing


Processed: 73/62178 Failed: 0 Avg: 19.06s ETA: 328.8h:   0%|          | 73/62178 [01:16<97:12:55,  5.64s/row] 

In [2]:
import pandas as pd
import ollama
import time
import re
from datetime import datetime
from pathlib import Path
from tqdm.notebook import tqdm
import concurrent.futures
import threading

# Configuration
model_name = "deepseek-r1:14b"
input_file = "/data/gregIB/issuebench/2_final_dataset/combined_prompts_issues_with_topics.csv"
safe_model_name = re.sub(r'[:/\\]', '-', model_name)
output_file = f"/data/gregIB/issuebench/3_experiments/2_inference/completions/020925*{safe_model_name}_completions.csv"

# Performance parameters
MAX_WORKERS = 4  # Number of parallel workers
CHUNK_SIZE = 50  # Save progress every 50 rows
UPDATE_FREQUENCY = 10  # Update progress every 10 rows

def call_ollama_model(prompt, model, max_retries=3):
    """Call Ollama model with simplified retry logic"""
    for attempt in range(max_retries):
        try:
            response = ollama.generate(
                model=model,
                prompt=prompt,
                options={'temperature': 1, 'top_p': 0.9, 'num_predict': 512}
            )
            return response['response'].strip()
        except Exception as e:
            if attempt == max_retries - 1:
                return f"[ERROR: {str(e)[:50]}]"
            time.sleep(1)  # Brief pause before retry

def process_batch(df, start_idx=0, resume=True):
    """Process dataframe batch with simplified parallel processing"""
    # Initialize tracking variables
    processed_count = start_idx
    total_rows = len(df)
    start_time = time.time()
    last_update_time = time.time()
    lock = threading.Lock()
    
    # Create output file if it doesn't exist
    if not Path(output_file).exists() or not resume:
        df['response_text'] = ''
        df['model'] = model_name
        df.to_csv(output_file, index=False)
    
    # Prepare tasks (only unprocessed rows)
    tasks = []
    for idx in range(start_idx, total_rows):
        prompt = str(df.iloc[idx]['prompt_text'])
        if not prompt or prompt.strip() == '' or prompt.lower() == 'nan':
            with lock:
                df.iloc[idx, df.columns.get_loc('response_text')] = '[EMPTY_PROMPT]'
                processed_count += 1
            continue
        
        tasks.append((idx, prompt))
    
    print(f"Processing {len(tasks)} tasks with {MAX_WORKERS} workers")
    
    # Process tasks in parallel
    with concurrent.futures.ThreadPoolExecutor(max_workers=MAX_WORKERS) as executor:
        future_to_idx = {
            executor.submit(call_ollama_model, prompt, model_name): idx 
            for idx, prompt in tasks
        }
        
        for future in tqdm(concurrent.futures.as_completed(future_to_idx), 
                          total=len(tasks), desc="Processing"):
            idx = future_to_idx[future]
            
            try:
                result = future.result()
                with lock:
                    df.iloc[idx, df.columns.get_loc('response_text')] = result
                    processed_count += 1
                    
                    # Update progress every UPDATE_FREQUENCY rows
                    if processed_count % UPDATE_FREQUENCY == 0:
                        elapsed = time.time() - start_time
                        rows_left = total_rows - processed_count
                        time_per_row = elapsed / processed_count
                        eta_seconds = rows_left * time_per_row
                        eta_str = str(datetime.now() + pd.Timedelta(seconds=eta_seconds))[:19]
                        
                        print(f"\nProcessed: {processed_count}/{total_rows} | "
                              f"ETA: {eta_str} | "
                              f"Time/row: {time_per_row:.2f}s")
                    
                    # Save checkpoint periodically
                    if processed_count % CHUNK_SIZE == 0:
                        df.to_csv(output_file, index=False)
                        print(f"Checkpoint saved at row {processed_count}")
                        
            except Exception as e:
                with lock:
                    df.iloc[idx, df.columns.get_loc('response_text')] = f'[ERROR: {str(e)[:50]}]'
                    processed_count += 1
    
    # Final save
    df.to_csv(output_file, index=False)
    total_time = time.time() - start_time
    print(f"Completed! Processed {processed_count} rows in {total_time:.2f} seconds "
          f"({total_time/processed_count:.2f}s per row)")

def run_processing(test_rows=None):
    """Main function to run processing"""
    # Load data
    df = pd.read_csv(input_file)
    if test_rows:
        df = df.head(test_rows)
    
    # Check for existing progress
    start_idx = 0
    if Path(output_file).exists():
        existing_df = pd.read_csv(output_file)
        if len(existing_df) == len(df):
            # Find first empty response
            for i, response in enumerate(existing_df['response_text']):
                if pd.isna(response) or response == '':
                    start_idx = i
                    break
            else:
                start_idx = len(df)  # All rows processed
                
            # Copy existing responses
            df['response_text'] = existing_df['response_text']
    
    print(f"Starting from row {start_idx} of {len(df)}")
    process_batch(df, start_idx)

# Run with a small test first, then full processing
run_processing(test_rows=20)  # Test with 20 rows
# run_processing()  # Process all rows

Starting from row 0 of 20
Processing 20 tasks with 4 workers


KeyboardInterrupt: 